In [1]:
# libraries for part 1
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
import re
import sys

pd.set_option('display.max_columns', 200)
plt.style.use("ggplot")

# Libraries for part 2
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# libraries part bert
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import RobertaTokenizer, RobertaModel

In [18]:
df = (
    pl.read_csv('./data/995,000_rows.csv', ignore_errors=True, n_rows=100_000)
    .select(['type', 'content'])
    .with_columns(pl.col('content').str.replace_all('\n', '').str.replace_all('\t', ''))
    .rename({'content':'title', 'type':'isfake'})
)

real_labels = ['reliable', 'political']
drop_labels = ['unknown', 'satire', 'bias', 'clickbait', 'unreliable', 'state', 'nan']

df = df.filter(
    ~pl.col('isfake').is_in(drop_labels)).with_columns(
    pl.when(pl.col('isfake').is_in(real_labels))
    .then(pl.lit('real'))
    .otherwise(pl.lit('fake'))
    .alias('label')
)

df = df.to_pandas().iloc[:25000]
df['isfake'] = df['label'].map(lambda x : 1 if x == 'fake' else 0)
df.drop(['label'], axis=1, inplace=True)
df

,isfake,title
0,0,Plus one article on Google Plus(Thanks to Ali ...
1,1,The Cost Of The Best Senate Banking Committee ...
2,0,WHEN Julia Geist was asked to draw a picture o...
3,1,– 100 Compiled Studies on Vaccine Dangers (Act...
4,0,IF YOU SPEND the majority of your waking hours...
...,...,...
24995,0,About the Author David Edwards has served as a...
24996,0,A 17-year-old man who told the British Olympic...
24997,0,"Apparel & Accessories | Wed Nov 9, 2016 | 2:13..."
24998,0,Faulk crouches just before the snap of the bal...


In [ ]:
class TitleDataset(Dataset):
    def __init__(self, df):
        self.titles = df['title'].astype('str').fillna('no title')
        self.targets = df['isfake']

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        return self.titles.iloc[idx], self.targets.iloc[idx]

class RobertaClassifier(nn.Module):
    def __init__(self, bertmodel='roberta-base'):
        super().__init__()
        self.tokenizer = RobertaTokenizer.from_pretrained(bertmodel)
        self.encoder = RobertaModel.from_pretrained(bertmodel)
        self.fc1 = nn.Linear(self.encoder.config.hidden_size, 256)
        self.fc2 = nn.Linear(256,128)
        self.fc3 = nn.Linear(128,32)
        self.fc4 = nn.Linear(32,16)
        self.fc5 = nn.Linear(16,1)

    def mean_pooling(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_embs = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embs.size()).float()
        vectors = (token_embs * mask).sum(1) / mask.sum(1)
        return vectors

    def forward(self, inputs):
        x = self.mean_pooling(inputs['input_ids'], inputs['attention_mask'])
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = self.fc5(x)
        return x.float()

    def predict(self, outputs):
        return (outputs > 0).float().squeeze(-1)

In [ ]:
model = RobertaClassifier()

In [ ]:
# training loop
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 1e-5},
    {'params': model.fc1.parameters(), 'lr': 1e-4},
])
dataset = TitleDataset(df)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

epochs = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device {device}.')
model.to(device)
losses = []
accs = []

for name, param in model.encoder.named_parameters():
    param.requires_grad = True
    
for e in range(epochs):
    running_loss = 0
    running_acc = 0
    for idx,(titles,labels) in enumerate(loader):
        inputs = model.tokenizer(list(titles), padding=True, truncation=True, max_length=64, return_tensors='pt')
        inputs = {k:v.to(device) for k,v in inputs.items()}

        optimizer.zero_grad()

        preds = model.forward(inputs)
        loss = criterion(preds, labels.float().unsqueeze(-1).to(device))
        loss.backward()

        optimizer.step()

        pred_labels = model.predict(preds)
        acc = float((pred_labels == labels.to(device)).float().mean())
        
        running_loss += loss.item()
        running_acc += acc
        sys.stdout.write(f'\rEpoch: {e+1}/{epochs}, Batch: {idx+1}/{len(loader)}, Loss: {loss.item():.4f}, Avg: {running_loss/(idx+1):.4f}, Acc: {acc:.4f}')
    losses.append(running_loss/(idx+1))
    accs.append(running_acc/(idx+1))

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.plot(losses, label='loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross Entropy Loss')
plt.title('RoBERTa Loss')
plt.legend
plt.subplot(1,2,2)
plt.plot(accs, label='acc', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('RoBERTa Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# read in data
df_test = pd.read_csv('10000_sample_rows_content.csv', nrows=10000).iloc[9000:10000]
df_test['content'] = df_test['content'].str.replace('\n','').str.replace('\t','')
df_test = df_test.rename(columns={'content':'title'})
df_test

In [ ]:
test_example = pd.DataFrame({'isfake':[1], 'title':['isreal declares random goat as the 1234 new prime minister of chicago'],'id':[333.0]})

df_test_with_example = pd.concat([df_test, test_example], axis=0)
df_test_with_example.tail()

In [ ]:
testset = TitleDataset(df_test)
testloader = DataLoader(testset, batch_size=10, shuffle=False, num_workers=2)

with torch.no_grad():
    losses = []
    accs = []
    f1s = []
    total_preds = []
    total_labels = []
    for idx,(titles,labels) in enumerate(testloader):
        inputs = model.tokenizer(list(titles), padding=True, truncation=True, max_length=64, return_tensors='pt')
        inputs = {k:v.to(device) for k,v in inputs.items()}

        preds = model.forward(inputs)
        loss = criterion(preds, labels.float().unsqueeze(-1).to(device))

        pred_labels = (preds > 0).float().squeeze(-1)

        total_labels += [int(l) for l in labels]
        total_preds += [int(p) for p in total_preds]

        acc = float((pred_labels == labels.to(device)).float().mean())
        f1 = f1_score(pred_labels.cpu().numpy(), labels.to(device).cpu().numpy())

        
        losses += [loss.item()]
        accs += [acc]
        f1s += [f1]
        sys.stdout.write(f'\rBatch: {idx+1}/{len(testloader)}, Loss: {loss.item():.4f}, Avg: {running_loss/(idx+1):.4f}, Acc: {acc:.4f}. F1: {f1:.4f}')


print(f'\nMean BCE Loss: {np.mean(losses):.4f}')
print(f'Mean Accuracy: {np.mean(accs):.4f}')
print(f'Mean F1: {np.mean(f1s):.4f}')
print(classification_report(total_preds, total_labels))



plt.figure(figsize=(12,6))
plt.subplot(1,3,1)
plt.hist(losses, label='loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross Entropy Loss')
plt.title('RoBERTa Loss')
plt.legend
plt.subplot(1,3,2)
plt.hist(accs, label='acc', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('RoBERTa Accuracy')
plt.legend()
plt.subplot(1,3,3)
plt.hist(accs, label='f1', color='green')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.title('RoBERTa F1')
plt.legend()
plt.tight_layout()
plt.show()